# Vector Database Learning Guide
A hands-on tour of embeddings, similarity search, and vector databases.

## Contents

|   | Topic | Key Concepts |
|---|-------|---------------|
| 1 | [What Are Embeddings & Vector DBs?](#1.-What-Are-Embeddings-&-Vector-DBs?) | vectors, semantic meaning, use cases |
| 2 | [Similarity Metrics](#2.-Similarity-Metrics) | cosine, euclidean, dot product |
| 3 | [Creating Embeddings](#3.-Creating-Embeddings) | `sentence-transformers`, text → vector |
| 4 | [Manual Vector Search (NumPy)](#4.-Manual-Vector-Search-(NumPy)) | brute-force search, ranking |
| 5 | [FAISS — Fast Similarity Search](#5.-FAISS-—-Fast-Similarity-Search) | `IndexFlatL2`, `IndexIVFFlat`, `IndexHNSW` |
| 6 | [ChromaDB — Local Vector Database](#6.-ChromaDB-—-Local-Vector-Database) | collections, add, query, metadata filtering |
| 7 | [Metadata Filtering & Advanced Queries](#7.-Metadata-Filtering-&-Advanced-Queries) | `where`, `where_document`, combining filters |
| 8 | [Indexing Strategies & Performance](#8.-Indexing-Strategies-&-Performance) | flat vs IVF vs HNSW, trade-offs |
| 9 | [Practical Example — Semantic Document Search](#9.-Practical-Example-—-Semantic-Document-Search) | end-to-end RAG-style retrieval |

## 1. What Are Embeddings & Vector DBs?

**Embeddings** convert data (text, images, audio) into dense numerical vectors that capture semantic meaning.
Similar items have vectors that are close together in this high-dimensional space.

**Vector databases** store and efficiently search these embeddings at scale.

### Common Use Cases
- **Semantic search** — find documents by meaning, not just keywords
- **Recommendation systems** — "users who liked X also liked Y"
- **RAG (Retrieval-Augmented Generation)** — feed relevant context to LLMs
- **Duplicate / anomaly detection** — find near-identical items
- **Image search** — find visually similar images

### How It Works
```
Text → Embedding Model → [0.12, -0.45, 0.78, ...] → Store in Vector DB
Query → Embedding Model → [0.11, -0.44, 0.79, ...] → Search Vector DB → Top-K results
```

## 2. Similarity Metrics

The three most common distance/similarity functions used in vector search.

In [1]:
import numpy as np

# Two sample vectors
a = np.array([1.0, 2.0, 3.0])
b = np.array([1.1, 2.1, 2.9])
c = np.array([5.0, -1.0, 0.5])   # intentionally different

# ── Cosine Similarity: measures angle between vectors (scale-invariant) ──────
#    1 = identical direction, 0 = orthogonal, -1 = opposite
def cosine_similarity(x, y):
    return np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))

print("Cosine Similarity:")
print(f"  a vs b (similar):  {cosine_similarity(a, b):.6f}")
print(f"  a vs c (different): {cosine_similarity(a, c):.6f}")

# ── Euclidean Distance (L2): straight-line distance ──────────────────────────
#    0 = identical, larger = more different
def euclidean_distance(x, y):
    return np.linalg.norm(x - y)

print("\nEuclidean Distance:")
print(f"  a vs b (similar):  {euclidean_distance(a, b):.6f}")
print(f"  a vs c (different): {euclidean_distance(a, c):.6f}")

# ── Dot Product: combines magnitude and direction ────────────────────────────
#    Higher = more similar (when vectors are normalized, equals cosine sim)
def dot_product(x, y):
    return np.dot(x, y)

print("\nDot Product:")
print(f"  a vs b (similar):  {dot_product(a, b):.6f}")
print(f"  a vs c (different): {dot_product(a, c):.6f}")

# ── When to use which? ────────────────────────────────────────────────────────
print("\n--- When to use each metric ---")
print("Cosine:     Best for text embeddings (direction matters, not magnitude)")
print("Euclidean:  Best when magnitude matters (e.g., user rating vectors)")
print("Dot Product: Best for normalized vectors or when magnitude encodes importance")

Cosine Similarity:
  a vs b (similar):  0.998930
  a vs c (different): 0.234738

Euclidean Distance:
  a vs b (similar):  0.173205
  a vs c (different): 5.590170

Dot Product:
  a vs b (similar):  14.000000
  a vs c (different): 4.500000

--- When to use each metric ---
Cosine:     Best for text embeddings (direction matters, not magnitude)
Euclidean:  Best when magnitude matters (e.g., user rating vectors)
Dot Product: Best for normalized vectors or when magnitude encodes importance


## 3. Creating Embeddings

We use `sentence-transformers` to convert text into dense vector embeddings.

```bash
pip install sentence-transformers
```

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a lightweight model (384-dim vectors)
model = SentenceTransformer('all-MiniLM-L6-v2')

# ── Encode single text ────────────────────────────────────────────────────────
text = "Machine learning is a subset of artificial intelligence."
embedding = model.encode(text)
print(f"Text: '{text}'")
print(f"Embedding shape: {embedding.shape}")
print(f"First 10 values: {embedding[:10].round(4)}")
print(f"Norm: {np.linalg.norm(embedding):.4f}")

# ── Encode multiple texts (batch) ────────────────────────────────────────────
sentences = [
    "I love programming in Python",
    "Python is my favorite language",
    "The weather is nice today",
    "I enjoy coding with Python",
    "It is raining outside",
]
embeddings = model.encode(sentences)
print(f"\nBatch shape: {embeddings.shape}")

# ── Check similarity between sentences ────────────────────────────────────────
from numpy import dot
from numpy.linalg import norm

def cos_sim(a, b):
    return dot(a, b) / (norm(a) * norm(b))

print("\nPairwise cosine similarities:")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        sim = cos_sim(embeddings[i], embeddings[j])
        print(f"  [{i}] vs [{j}]: {sim:.4f}  |  '{sentences[i][:35]}' vs '{sentences[j][:35]}'")

## 4. Manual Vector Search (NumPy)

Before using a vector DB, understand brute-force search — it's the baseline.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# ── Document corpus ───────────────────────────────────────────────────────────
documents = [
    "Python is a versatile programming language",
    "Machine learning models need training data",
    "Neural networks are inspired by the human brain",
    "Data preprocessing is crucial for ML pipelines",
    "Natural language processing handles text data",
    "Computer vision deals with image recognition",
    "Deep learning uses multiple neural network layers",
    "Pandas is great for data manipulation in Python",
    "Scikit-learn provides simple ML algorithms",
    "TensorFlow and PyTorch are popular DL frameworks",
]

# Encode all documents
doc_embeddings = model.encode(documents)
print(f"Corpus: {len(documents)} documents, {doc_embeddings.shape[1]}-dim embeddings")

# ── Search function ───────────────────────────────────────────────────────────
def search(query, doc_embeddings, documents, top_k=3):
    query_emb = model.encode([query])[0]
    # Compute cosine similarity with all documents
    similarities = np.dot(doc_embeddings, query_emb) / (
        np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_emb)
    )
    # Get top-k indices
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [(documents[i], similarities[i]) for i in top_indices]

# ── Try different queries ─────────────────────────────────────────────────────
queries = [
    "How do I analyze data with Python?",
    "What is deep learning?",
    "image classification techniques",
]

for query in queries:
    print(f"\nQuery: '{query}'")
    results = search(query, doc_embeddings, documents, top_k=3)
    for rank, (doc, score) in enumerate(results, 1):
        print(f"  {rank}. [{score:.4f}] {doc}")

## 5. FAISS — Fast Similarity Search

Facebook AI Similarity Search — optimized C++ library for billion-scale vector search.

```bash
pip install faiss-cpu
```

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

documents = [
    "Python is a versatile programming language",
    "Machine learning models need training data",
    "Neural networks are inspired by the human brain",
    "Data preprocessing is crucial for ML pipelines",
    "Natural language processing handles text data",
    "Computer vision deals with image recognition",
    "Deep learning uses multiple neural network layers",
    "Pandas is great for data manipulation in Python",
    "Scikit-learn provides simple ML algorithms",
    "TensorFlow and PyTorch are popular DL frameworks",
]

# Encode and convert to float32 (FAISS requirement)
embeddings = model.encode(documents).astype('float32')
dim = embeddings.shape[1]

# ── IndexFlatL2: exact brute-force search (L2 distance) ──────────────────────
index_flat = faiss.IndexFlatL2(dim)
index_flat.add(embeddings)
print(f"Index size: {index_flat.ntotal} vectors, {dim} dimensions")

# Search
query = "How do neural networks work?"
query_emb = model.encode([query]).astype('float32')

distances, indices = index_flat.search(query_emb, k=3)  # top-3

print(f"\nQuery: '{query}'")
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"  {rank}. [L2={dist:.4f}] {documents[idx]}")

In [ ]:
import faiss
import numpy as np

# ── Comparing FAISS index types at larger scale ───────────────────────────────
np.random.seed(42)
dim = 128
n_vectors = 50_000
n_queries = 100

# Random vectors (simulating embeddings)
data = np.random.random((n_vectors, dim)).astype('float32')
queries = np.random.random((n_queries, dim)).astype('float32')

# --- IndexFlatL2: exact, brute-force (baseline) ---
index_flat = faiss.IndexFlatL2(dim)
index_flat.add(data)

import time
t0 = time.time()
D_flat, I_flat = index_flat.search(queries, k=10)
t_flat = time.time() - t0
print(f"IndexFlatL2:   {t_flat*1000:.1f} ms  (exact, no training needed)")

# --- IndexIVFFlat: approximate, partitions space into clusters ---
nlist = 100  # number of clusters
quantizer = faiss.IndexFlatL2(dim)
index_ivf = faiss.IndexIVFFlat(quantizer, dim, nlist)
index_ivf.train(data)      # IVF requires training
index_ivf.add(data)
index_ivf.nprobe = 10      # search 10 out of 100 clusters

t0 = time.time()
D_ivf, I_ivf = index_ivf.search(queries, k=10)
t_ivf = time.time() - t0

# Recall: how many of the true top-10 did IVF find?
recall = np.mean([len(set(I_flat[i]) & set(I_ivf[i])) / 10 for i in range(n_queries)])
print(f"IndexIVFFlat:  {t_ivf*1000:.1f} ms  (recall@10={recall:.2%}, nprobe=10)")

# --- IndexHNSWFlat: graph-based approximate search ---
index_hnsw = faiss.IndexHNSWFlat(dim, 32)  # 32 = M (connections per node)
index_hnsw.add(data)

t0 = time.time()
D_hnsw, I_hnsw = index_hnsw.search(queries, k=10)
t_hnsw = time.time() - t0

recall_hnsw = np.mean([len(set(I_flat[i]) & set(I_hnsw[i])) / 10 for i in range(n_queries)])
print(f"IndexHNSWFlat: {t_hnsw*1000:.1f} ms  (recall@10={recall_hnsw:.2%}, M=32)")

print(f"\nTotal vectors: {n_vectors:,}  |  Dimension: {dim}  |  Queries: {n_queries}")

## 6. ChromaDB — Local Vector Database

ChromaDB is an open-source, developer-friendly vector database that runs locally.
It handles embedding, storage, and search in one package.

```bash
pip install chromadb
```

In [ ]:
import chromadb

# ── Create an in-memory client (ephemeral, for experimentation) ──────────────
client = chromadb.Client()

# ── Create a collection (like a table) ────────────────────────────────────────
# ChromaDB uses its own default embedding function (all-MiniLM-L6-v2)
collection = client.create_collection(
    name="my_documents",
    metadata={"hnsw:space": "cosine"}  # distance metric: cosine, l2, or ip
)
print(f"Collection: '{collection.name}'")
print(f"Count: {collection.count()}")

# ── Add documents ─────────────────────────────────────────────────────────────
collection.add(
    documents=[
        "Python is a versatile programming language used in web and data science",
        "Machine learning models require large amounts of training data",
        "Neural networks are computational models inspired by the human brain",
        "Data preprocessing is a crucial step in any ML pipeline",
        "Natural language processing enables computers to understand text",
        "Computer vision allows machines to interpret visual information",
        "Deep learning uses multiple layers to learn hierarchical features",
        "Pandas library simplifies data manipulation and analysis in Python",
        "Scikit-learn provides efficient tools for predictive data analysis",
        "TensorFlow and PyTorch are the most popular deep learning frameworks",
    ],
    ids=[f"doc_{i}" for i in range(10)],  # unique IDs required
    metadatas=[
        {"category": "programming", "level": "beginner"},
        {"category": "ml", "level": "intermediate"},
        {"category": "dl", "level": "intermediate"},
        {"category": "ml", "level": "beginner"},
        {"category": "nlp", "level": "intermediate"},
        {"category": "cv", "level": "intermediate"},
        {"category": "dl", "level": "advanced"},
        {"category": "programming", "level": "beginner"},
        {"category": "ml", "level": "beginner"},
        {"category": "dl", "level": "intermediate"},
    ]
)
print(f"Added documents. Count: {collection.count()}")

In [ ]:
# ── Query the collection ──────────────────────────────────────────────────────
results = collection.query(
    query_texts=["How do neural networks learn?"],
    n_results=3
)

print("Query: 'How do neural networks learn?'\n")
for i in range(len(results['ids'][0])):
    print(f"  {i+1}. [{results['distances'][0][i]:.4f}] {results['documents'][0][i]}")
    print(f"     ID: {results['ids'][0][i]}  |  metadata: {results['metadatas'][0][i]}")

# ── Multiple queries at once ──────────────────────────────────────────────────
print("\n--- Batch query ---")
batch_results = collection.query(
    query_texts=[
        "Python data analysis",
        "image recognition AI",
    ],
    n_results=2
)

for q_idx, query in enumerate(["Python data analysis", "image recognition AI"]):
    print(f"\nQuery: '{query}'")
    for i in range(len(batch_results['ids'][q_idx])):
        print(f"  {i+1}. {batch_results['documents'][q_idx][i]}")

In [ ]:
# ── CRUD operations ───────────────────────────────────────────────────────────

# Get documents by ID
result = collection.get(ids=["doc_0", "doc_3"])
print("Get by ID:")
for doc, meta in zip(result['documents'], result['metadatas']):
    print(f"  {doc}  |  {meta}")

# Update a document
collection.update(
    ids=["doc_0"],
    documents=["Python is an amazing language for web development, data science, and AI"],
    metadatas=[{"category": "programming", "level": "beginner", "updated": True}]
)
print("\nUpdated doc_0:")
print(collection.get(ids=["doc_0"])['documents'])

# Upsert: update if exists, insert if not
collection.upsert(
    ids=["doc_10"],
    documents=["Reinforcement learning trains agents through reward signals"],
    metadatas=[{"category": "rl", "level": "advanced"}]
)
print(f"\nAfter upsert, count: {collection.count()}")

# Delete
collection.delete(ids=["doc_10"])
print(f"After delete, count: {collection.count()}")

## 7. Metadata Filtering & Advanced Queries

Filter results by metadata fields to narrow down the search space.

In [ ]:
# ── Filter by metadata ────────────────────────────────────────────────────────

# Only search within "dl" (deep learning) category
results = collection.query(
    query_texts=["How do models learn?"],
    n_results=5,
    where={"category": "dl"}
)
print("Query + filter (category=dl):")
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(f"  {doc}  |  {meta}")

# ── Comparison operators: $eq, $ne, $gt, $gte, $lt, $lte, $in, $nin ──────────
results = collection.query(
    query_texts=["data analysis tools"],
    n_results=5,
    where={"level": {"$eq": "beginner"}}
)
print("\nQuery + filter (level=beginner):")
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(f"  {doc}  |  {meta}")

# ── $and / $or logical operators ──────────────────────────────────────────────
results = collection.query(
    query_texts=["AI techniques"],
    n_results=5,
    where={
        "$or": [
            {"category": "dl"},
            {"category": "ml"}
        ]
    }
)
print("\nQuery + filter (category=dl OR category=ml):")
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(f"  [{meta['category']}] {doc}")

# ── Filter by document content ────────────────────────────────────────────────
results = collection.query(
    query_texts=["programming"],
    n_results=5,
    where_document={"$contains": "Python"}
)
print("\nQuery + where_document (contains 'Python'):")
for doc in results['documents'][0]:
    print(f"  {doc}")

## 8. Indexing Strategies & Performance

Understanding trade-offs between speed, accuracy, and memory.

| Index Type | Speed | Accuracy | Memory | Training | Best For |
|------------|-------|----------|--------|----------|----------|
| **Flat (brute-force)** | Slow | Exact | High | No | < 100K vectors |
| **IVF (Inverted File)** | Fast | ~95-99% | Medium | Yes | 100K–10M vectors |
| **HNSW (Graph-based)** | Very fast | ~95-99% | High | No | < 10M, low latency |
| **PQ (Product Quantization)** | Very fast | ~90-95% | Low | Yes | > 10M, memory-limited |

In [ ]:
import faiss
import numpy as np
import time

np.random.seed(42)
dim = 128
n_vectors = 100_000

data = np.random.random((n_vectors, dim)).astype('float32')
query = np.random.random((1, dim)).astype('float32')

# ── Flat: exact baseline ──────────────────────────────────────────────────────
index_flat = faiss.IndexFlatL2(dim)
index_flat.add(data)
t0 = time.time()
D_exact, I_exact = index_flat.search(query, k=10)
t_flat = (time.time() - t0) * 1000

# ── IVF: cluster-based approximate ───────────────────────────────────────────
nlist = 256
quantizer = faiss.IndexFlatL2(dim)
index_ivf = faiss.IndexIVFFlat(quantizer, dim, nlist)
index_ivf.train(data)
index_ivf.add(data)

results = {}
for nprobe in [1, 5, 10, 50]:
    index_ivf.nprobe = nprobe
    t0 = time.time()
    D_ivf, I_ivf = index_ivf.search(query, k=10)
    t_ivf = (time.time() - t0) * 1000
    recall = len(set(I_exact[0]) & set(I_ivf[0])) / 10
    results[nprobe] = (t_ivf, recall)

print(f"{'Index':<25} {'Time (ms)':>10} {'Recall@10':>10}")
print("-" * 47)
print(f"{'Flat (exact)':<25} {t_flat:>10.2f} {'100.0%':>10}")
for nprobe, (t, r) in results.items():
    print(f"{'IVF nprobe=' + str(nprobe):<25} {t:>10.2f} {r:>9.0%}")

# ── HNSW ──────────────────────────────────────────────────────────────────────
for M in [16, 32, 64]:
    index_hnsw = faiss.IndexHNSWFlat(dim, M)
    index_hnsw.add(data)
    t0 = time.time()
    D_h, I_h = index_hnsw.search(query, k=10)
    t_h = (time.time() - t0) * 1000
    recall = len(set(I_exact[0]) & set(I_h[0])) / 10
    print(f"{'HNSW M=' + str(M):<25} {t_h:>10.2f} {recall:>9.0%}")

print(f"\nDataset: {n_vectors:,} vectors x {dim} dims")

## 9. Practical Example — Semantic Document Search

End-to-end example: index a corpus, search semantically, and retrieve relevant context (RAG-style).

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# ── Prepare a mini knowledge base ─────────────────────────────────────────────
knowledge_base = [
    {
        "text": "Python was created by Guido van Rossum and first released in 1991. It emphasizes code readability with its notable use of significant indentation.",
        "source": "wiki", "topic": "python"
    },
    {
        "text": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.",
        "source": "textbook", "topic": "ml"
    },
    {
        "text": "A neural network consists of layers of interconnected nodes. Each connection has a weight that is adjusted during training through backpropagation.",
        "source": "textbook", "topic": "dl"
    },
    {
        "text": "Transfer learning is a technique where a model trained on one task is reused as the starting point for a model on a different task.",
        "source": "paper", "topic": "ml"
    },
    {
        "text": "Vector databases store high-dimensional vectors and enable fast similarity search. They are essential for modern AI applications like RAG.",
        "source": "blog", "topic": "database"
    },
    {
        "text": "FAISS is a library developed by Facebook AI Research for efficient similarity search and clustering of dense vectors.",
        "source": "docs", "topic": "database"
    },
    {
        "text": "Transformers are a neural network architecture that uses self-attention mechanisms. They power models like BERT, GPT, and T5.",
        "source": "paper", "topic": "dl"
    },
    {
        "text": "Retrieval-Augmented Generation (RAG) combines a retrieval system with a language model to generate more accurate and grounded responses.",
        "source": "paper", "topic": "llm"
    },
    {
        "text": "Embeddings are dense vector representations that capture semantic meaning. Similar concepts have vectors that are close together in the embedding space.",
        "source": "textbook", "topic": "ml"
    },
    {
        "text": "ChromaDB is an open-source embedding database designed for AI applications. It provides simple APIs for storing, searching, and filtering embeddings.",
        "source": "docs", "topic": "database"
    },
]

# ── Index into ChromaDB ───────────────────────────────────────────────────────
client = chromadb.Client()
collection = client.create_collection(name="knowledge_base", metadata={"hnsw:space": "cosine"})

collection.add(
    documents=[item["text"] for item in knowledge_base],
    ids=[f"kb_{i}" for i in range(len(knowledge_base))],
    metadatas=[{"source": item["source"], "topic": item["topic"]} for item in knowledge_base],
)
print(f"Indexed {collection.count()} documents into ChromaDB")

In [ ]:
# ── Semantic search function ──────────────────────────────────────────────────
def semantic_search(query, n_results=3, topic_filter=None):
    """Search the knowledge base and return formatted results."""
    kwargs = {
        "query_texts": [query],
        "n_results": n_results,
    }
    if topic_filter:
        kwargs["where"] = {"topic": topic_filter}

    results = collection.query(**kwargs)

    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    if topic_filter:
        print(f"Filter: topic={topic_filter}")
    print(f"{'='*70}")

    for i in range(len(results['ids'][0])):
        score = results['distances'][0][i]
        doc   = results['documents'][0][i]
        meta  = results['metadatas'][0][i]
        print(f"\n  [{i+1}] Score: {score:.4f}  |  source: {meta['source']}  |  topic: {meta['topic']}")
        print(f"      {doc}")

# ── Run queries ───────────────────────────────────────────────────────────────
semantic_search("What is RAG and how does it work?")
semantic_search("Tell me about Python programming language")
semantic_search("How do transformers and attention mechanisms work?")
semantic_search("vector similarity search tools", topic_filter="database")

# ── RAG-style context building ────────────────────────────────────────────────
print("\n" + "="*70)
print("RAG-style context for: 'Explain how modern AI search works'")
print("="*70)

results = collection.query(query_texts=["Explain how modern AI search works"], n_results=3)
context = "\n".join(f"- {doc}" for doc in results['documents'][0])

prompt = f"""Use the following context to answer the question.

Context:
{context}

Question: Explain how modern AI search works.
Answer:"""
print(prompt)
print("\n(This prompt would be sent to an LLM like Claude for final answer generation)")

## Bonus: Persistent Storage with ChromaDB

By default ChromaDB is in-memory. Use `PersistentClient` to save to disk.

In [ ]:
import chromadb
import os
import shutil

# ── Persistent client saves data to disk ──────────────────────────────────────
db_path = "./chroma_persist_demo"
client = chromadb.PersistentClient(path=db_path)

col = client.get_or_create_collection("persistent_demo")
col.add(
    documents=["This data survives restarts!", "Persistent vector storage."],
    ids=["p1", "p2"]
)
print(f"Saved {col.count()} docs to {db_path}")
print(f"Files on disk: {os.listdir(db_path)}")

# Simulate restart: create a new client pointing to same path
client2 = chromadb.PersistentClient(path=db_path)
col2 = client2.get_collection("persistent_demo")
print(f"\nAfter 'restart', count: {col2.count()}")
print("Documents:", col2.get()['documents'])

# Clean up demo
shutil.rmtree(db_path)
print(f"\nCleaned up {db_path}")